# Day 09. Exercise 00
# Regularization

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import joblib

## 1. Preprocessing

1. Read the file `dayofweek.csv` that you used in the previous day to a dataframe.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [2]:
df = pd.read_csv('dayofweek.csv')
print(df.shape)
print(df.dtypes.to_list())
df.head()

(1686, 44)
[dtype('int64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64')]


,dayofweek,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,4,-0.788667,-2.562352,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,4,-0.756764,-2.562352,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,4,-0.724861,-2.562352,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,4,-0.692958,-2.562352,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,4,-0.661055,-2.562352,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [3]:
random_state=21
test_size=0.2
X = df.drop('dayofweek', axis=1)
y = df['dayofweek']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)
print('X_train shape: ', X_train.shape)
print('y_train shape: ', y_train.shape)
print('X_test shape : ', X_test.shape)
print('y_test shape : ', y_test.shape)

X_train shape:  (1348, 43)
y_train shape:  (1348,)
X_test shape :  (338, 43)
y_test shape :  (338,)


## 2. Logreg regularization

### a. Default regularization

1. Train a baseline model with the only parameters `random_state=21`, `fit_intercept=False`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model


The result of the code where you trained and evaluated the baseline model should be exactly like this (use `%%time` to get the info about how long it took to run the cell):

```
train -  0.62902   |   valid -  0.59259
train -  0.64633   |   valid -  0.62963
train -  0.63479   |   valid -  0.56296
train -  0.65622   |   valid -  0.61481
train -  0.63397   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64138   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63674   |   valid -  0.62687
Average accuracy on crossval is 0.60165
Std is 0.02943
```

In [4]:
model1 = LogisticRegression(random_state=random_state, fit_intercept=False)
model1.fit(X_train, y_train)

n_splits=10
cv = StratifiedKFold(n_splits=n_splits)

In [5]:
def train_and_evaluate(model, X_train, y_train):

    train_scores = []
    val_scores = []

    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model.fit(X_tr, y_tr)

        train_acc = accuracy_score(y_tr, model.predict(X_tr))
        val_acc = accuracy_score(y_val, model.predict(X_val))
        train_scores.append(train_acc)
        val_scores.append(val_acc)

        print(f"train -  {train_acc:.5f}   |   valid -  {val_acc:.5f}")

    avg_val_acc = np.mean(val_scores)
    std_val_acc = np.std(val_scores)

    print(f"Average accuracy on crossval is {avg_val_acc:.5f}")
    print(f"Std is {std_val_acc:.5f}")

In [6]:
%time train_and_evaluate(model1, X_train, y_train)

train -  0.62902   |   valid -  0.59259
train -  0.64633   |   valid -  0.62963
train -  0.63479   |   valid -  0.56296
train -  0.65622   |   valid -  0.61481
train -  0.63397   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64138   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63674   |   valid -  0.62687
Average accuracy on crossval is 0.60165
Std is 0.02943
CPU times: user 481 ms, sys: 328 ms, total: 810 ms
Wall time: 708 ms


### b. Optimizing regularization parameters

1. In the cells below try different values of penalty: `none`, `l1`, `l2` – you can change the values of solver too.

In [7]:
def short_output_train_and_evaluate(model, X_train, y_train):

    train_scores = []
    val_scores = []

    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model.fit(X_tr, y_tr)

        train_acc = accuracy_score(y_tr, model.predict(X_tr))
        val_acc = accuracy_score(y_val, model.predict(X_val))
        train_scores.append(train_acc)
        val_scores.append(val_acc)

    avg_val_acc = np.mean(val_scores)
    std_val_acc = np.std(val_scores)

    print(f"Average accuracy on crossval is {avg_val_acc:.5f}, std is {std_val_acc:.5f}.")
    
penalties = ['none', 'l1', 'l2']
solvers = ['lbfgs', 'liblinear', 'saga']

for penalty in penalties:
    for solver in solvers:
        model = LogisticRegression(
            random_state=random_state, 
            fit_intercept=False, 
            penalty=penalty, 
            solver=solver,
            max_iter=5000
        )

        if (penalty=='none' and solver=='liblinear') or (penalty=='l1' and solver=='lbfgs'):
            continue
        else: 
            print(f"\nTesting penalty={penalty}, solver={solver}:")
            short_output_train_and_evaluate(model, X_train, y_train)



Testing penalty=none, solver=lbfgs:
Average accuracy on crossval is 0.62462, std is 0.03379.

Testing penalty=none, solver=saga:
Average accuracy on crossval is 0.62462, std is 0.03379.

Testing penalty=l1, solver=liblinear:
Average accuracy on crossval is 0.58903, std is 0.03129.

Testing penalty=l1, solver=saga:
Average accuracy on crossval is 0.59646, std is 0.02848.

Testing penalty=l2, solver=lbfgs:
Average accuracy on crossval is 0.60165, std is 0.02943.

Testing penalty=l2, solver=liblinear:
Average accuracy on crossval is 0.58160, std is 0.02532.

Testing penalty=l2, solver=saga:
Average accuracy on crossval is 0.60165, std is 0.02943.


## 3. SVM regularization

### a. Default regularization

1. Train a baseline model with the only parameters `probability=True`, `kernel='linear'`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [8]:
model2 = SVC(probability=True, kernel='linear', random_state=random_state)
model2.fit(X_train, y_train)

n_splits=10
cv = StratifiedKFold(n_splits=n_splits)

In [9]:
%time train_and_evaluate(model2, X_train, y_train)

train -  0.70486   |   valid -  0.65926
train -  0.69662   |   valid -  0.75556
train -  0.69415   |   valid -  0.62222
train -  0.70239   |   valid -  0.65185
train -  0.69085   |   valid -  0.65185
train -  0.68920   |   valid -  0.64444
train -  0.69250   |   valid -  0.72593
train -  0.70074   |   valid -  0.62222
train -  0.69605   |   valid -  0.61940
train -  0.71087   |   valid -  0.63433
Average accuracy on crossval is 0.65871
Std is 0.04359
CPU times: user 2.28 s, sys: 158 ms, total: 2.44 s
Wall time: 2.46 s


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `C`.

In [10]:
Cs = [100, 10, 1, 0.1, 0.01, 0.001]

for C in Cs:
    model = SVC(
        probability=True,
        kernel='linear',
        random_state=random_state,
        C=C
    )

    print(f"\nTesting C={C}:")
    short_output_train_and_evaluate(model, X_train, y_train)


Testing C=100:
Average accuracy on crossval is 0.75589, std is 0.03550.

Testing C=10:
Average accuracy on crossval is 0.72771, std is 0.04417.

Testing C=1:
Average accuracy on crossval is 0.65871, std is 0.04359.

Testing C=0.1:
Average accuracy on crossval is 0.56230, std is 0.02177.

Testing C=0.01:
Average accuracy on crossval is 0.37534, std is 0.01848.

Testing C=0.001:
Average accuracy on crossval is 0.23443, std is 0.00397.


## 4. Tree

### a. Default regularization

1. Train a baseline model with the only parameter `max_depth=10` and `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [11]:
model3 = DecisionTreeClassifier(max_depth=10, random_state=random_state)
model3.fit(X_train, y_train)

n_splits=10
cv = StratifiedKFold(n_splits=n_splits)

In [12]:
%time train_and_evaluate(model3, X_train, y_train)

train -  0.81039   |   valid -  0.74074
train -  0.77741   |   valid -  0.74074
train -  0.83347   |   valid -  0.70370
train -  0.79720   |   valid -  0.76296
train -  0.82440   |   valid -  0.75556
train -  0.80379   |   valid -  0.68889
train -  0.80709   |   valid -  0.76296
train -  0.80132   |   valid -  0.65926
train -  0.80807   |   valid -  0.75373
train -  0.80478   |   valid -  0.68657
Average accuracy on crossval is 0.72551
Std is 0.03562
CPU times: user 41.1 ms, sys: 25.9 ms, total: 67.1 ms
Wall time: 88.3 ms


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `max_depth`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [13]:
max_depth_list = range(5, 13)

for max_depth in max_depth_list:
    model = DecisionTreeClassifier(
        random_state=random_state,
        max_depth=max_depth
    )

    print(f"\nTesting max_depth={max_depth}:")
    short_output_train_and_evaluate(model, X_train, y_train)


Testing max_depth=5:
Average accuracy on crossval is 0.54301, std is 0.02423.

Testing max_depth=6:
Average accuracy on crossval is 0.59570, std is 0.03040.

Testing max_depth=7:
Average accuracy on crossval is 0.64989, std is 0.03971.

Testing max_depth=8:
Average accuracy on crossval is 0.66098, std is 0.03700.

Testing max_depth=9:
Average accuracy on crossval is 0.70325, std is 0.04068.

Testing max_depth=10:
Average accuracy on crossval is 0.72551, std is 0.03562.

Testing max_depth=11:
Average accuracy on crossval is 0.76999, std is 0.03790.

Testing max_depth=12:
Average accuracy on crossval is 0.80413, std is 0.03947.


## 5. Random forest

### a. Default regularization

1. Train a baseline model with the only parameters `n_estimators=50`, `max_depth=14`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [14]:
model4 = RandomForestClassifier(n_estimators=50, max_depth=14, random_state=random_state)
model4.fit(X_train, y_train)

n_splits=10
cv = StratifiedKFold(n_splits=n_splits)

In [15]:
%time train_and_evaluate(model4, X_train, y_train)

train -  0.96455   |   valid -  0.88148
train -  0.96208   |   valid -  0.91852
train -  0.96785   |   valid -  0.86667
train -  0.96455   |   valid -  0.89630
train -  0.96538   |   valid -  0.91111
train -  0.96538   |   valid -  0.88148
train -  0.97115   |   valid -  0.91852
train -  0.96867   |   valid -  0.85185
train -  0.97364   |   valid -  0.88060
train -  0.97941   |   valid -  0.86567
Average accuracy on crossval is 0.88722
Std is 0.02204
CPU times: user 717 ms, sys: 24.4 ms, total: 741 ms
Wall time: 762 ms


### b. Optimizing regularization parameters

1. In the new cells try different values of the parameters `max_depth` and `n_estimators`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [16]:
max_depth_list = range(14, 17)
n_estimators_list = [100, 200, 300]

for max_depth in max_depth_list:
    for n_estimators in  n_estimators_list:
        model = RandomForestClassifier(
            random_state=random_state,
            max_depth=max_depth,
            n_estimators=n_estimators
        )

        print(f"\nTesting max_depth={max_depth} and n_estimators={n_estimators}:")
        short_output_train_and_evaluate(model, X_train, y_train)


Testing max_depth=14 and n_estimators=100:
Average accuracy on crossval is 0.88574, std is 0.01773.

Testing max_depth=14 and n_estimators=200:
Average accuracy on crossval is 0.88870, std is 0.01805.

Testing max_depth=14 and n_estimators=300:
Average accuracy on crossval is 0.88870, std is 0.01464.

Testing max_depth=15 and n_estimators=100:
Average accuracy on crossval is 0.89612, std is 0.01795.

Testing max_depth=15 and n_estimators=200:
Average accuracy on crossval is 0.89463, std is 0.01636.

Testing max_depth=15 and n_estimators=300:
Average accuracy on crossval is 0.89538, std is 0.01479.

Testing max_depth=16 and n_estimators=100:
Average accuracy on crossval is 0.89391, std is 0.01882.

Testing max_depth=16 and n_estimators=200:
Average accuracy on crossval is 0.89834, std is 0.02006.

Testing max_depth=16 and n_estimators=300:
Average accuracy on crossval is 0.90130, std is 0.02036.


## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.
3. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your test dataset).
4. Save the model.

In [17]:
best_model = RandomForestClassifier(random_state=random_state,
                                    max_depth=16,
                                    n_estimators=300)
best_model.fit(X_train, y_train)
predict = best_model.predict(X_test)
acc = accuracy_score(predict, y_test)
print("Accuracy_score_4_best_model: ", acc)

Accuracy_score_4_best_model:  0.9112426035502958


In [18]:
df_errors = pd.DataFrame({'y_true': y_test, 'y_pred': predict})

In [19]:
errors_per_class = (df_errors['y_true'] != df_errors['y_pred']).groupby(df_errors['y_true']).sum()
total_per_class = df_errors['y_true'].value_counts()
error_percentage = (errors_per_class / total_per_class * 100).sort_values(ascending=False)

print(error_percentage)

0    25.925926
4    14.285714
1    10.909091
5     9.259259
2     6.666667
6     5.633803
3     3.750000
dtype: float64


In [20]:
joblib.dump(best_model, 'best_model.joblib')

['best_model.joblib']